# F2 · Rare classes: weights, undersampling, oversampling or just the threshold?

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare five strategies for a rare event (about 3.5% positives): base model with threshold 0.5, `class_weight="balanced"`, random undersampling and oversampling (on the training set only) and base model with **threshold chosen
in cross-validation**. Measure average precision, AUC, F1 and Brier score on 20,000 new examples, averaged over 10 datasets.

## Dataset

`generate(seed)` (given): 23,000 examples, 12 features (5 informative), about 3% positives; training = the first 3000 (about 105 positives), test = the other 20,000.

## Educational purpose

Replace intuition ("with imbalanced classes you must rebalance") with a measurement. You will find that the rebalancing techniques **shift the effective threshold**, they don't improve the ordering, and they **ruin** the probabilities.

## Task

1. Write `best_threshold(y, p)`: the threshold (grid `np.linspace(0.02, 0.9, 89)`) that maximizes F1.
2. For each seed from 0 to 9 and for each of the five strategies (`base`, `balanced`, `undersampling`, `oversampling`, `cv threshold`) compute on the test set: `ap`, `auc`, `f1`, `recall`, `precision` (threshold 0.5 except `cv threshold`), `brier` and `mean_p`. Undersampling: all the positives + the same number of random negatives; oversampling: all the negatives + positives repeated at random until they match (`rng = np.random.default_rng(seed)`). `cv threshold`: base model with the threshold chosen with `best_threshold` on the out-of-sample probabilities of `cross_val_predict` (5 stratified folds) on the training set.
3. Save `table` (mean over the seeds, rows: strategies) and the answers: `best_f1` (strategy with maximum mean F1), `best_brier` (minimum Brier) and `balanced_mean_p` (mean probability predicted by the `balanced` strategy).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(seed):
    X, y = make_classification(n_samples=23000, n_features=12, n_informative=5, n_redundant=0, weights=[0.97], class_sep=0.9, flip_y=0.01, random_state=seed)
    return X[:3000], y[:3000], X[3000:], y[3000:]


def model(**kw):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, **kw))

## Your work

In [ ]:
def best_threshold(y, p):
    ...

In [ ]:
rows = []
for seed in range(10):
    X_tr, y_tr, X_te, y_te = generate(seed)
    rng = np.random.default_rng(seed)
    ...
table = pd.DataFrame(rows).groupby("strategy").mean().drop(columns="seed")
table.round(3)

In [ ]:
best_f1 = None
best_brier = None
balanced_mean_p = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`best_threshold`: `ts = np.linspace(0.02, 0.9, 89); ts[np.argmax([f1_score(y, p >= t) for t in ts])]`. Undersampling: `pos = np.where(y_tr == 1)[0]; neg = np.where(y_tr == 0)[0]; idx = np.r_[pos, rng.choice(neg, len(pos), replace=False)]`. Oversampling: `idx = np.r_[neg, rng.choice(pos, len(neg))]`. Metrics: a function `row(name, m, X_fit, y_fit, threshold=0.5)` avoids repeating the code.'):
    check(set(table.index) == {"base", "balanced", "undersampling", "oversampling", "cv threshold"} and list(table.columns) == ["ap", "auc", "f1", "recall", "precision", "brier", "mean_p"], "five strategies and seven metrics", "columns: ap, auc, f1, recall, precision, brier, mean_p")
    check(best_f1 == "cv threshold" and table.loc["cv threshold", "f1"] > table.loc["base", "f1"] + 0.08, "the threshold chosen in cross-validation has the best F1 of all (and by over 0.08 compared to base)", "compare the f1 column")
    check(table.loc[["balanced", "undersampling", "oversampling"], "recall"].min() > 0.6 and table.loc[["balanced", "undersampling", "oversampling"], "precision"].max() < 0.2, "the three rebalancings have high recall (above 0.6) and low precision (below 0.2)", "they are shifts of the effective threshold")
    check(abs(table["auc"].max() - table["auc"].min()) < 0.03, "the AUC is practically the same (difference below 0.03)", "rebalancing does not change the ordering much")
    check(table.loc["base", "ap"] > table.loc[["balanced", "undersampling", "oversampling"], "ap"].max() + 0.05, "the average precision of the base model is better by 0.05 than any rebalancing", "AP: base model against rebalanced ones")
    check(best_brier in ("base", "cv threshold") and balanced_mean_p > 0.25 and table.loc["base", "mean_p"] < 0.05, "the best Brier is that of the base model; `balanced` predicts on average over 0.25 against a prevalence of 3.5%", "minimum brier and mean_p")
done()

---
The commented solution is in `solutions/F_compare/F2_strategies_for_rare_classes.ipynb`. Open it after you pass the check, or after you have really tried.